LPG储罐虚拟传感器 - 使用Conformal Prediction进行不确定性量化

Conformal Prediction提供严格的统计覆盖率保证

**新增功能**: 综合可信度指数(CTI)计算

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import spearmanr  # 新增: 用于计算Spearman相关系数
import matplotlib.pyplot as plt
import copy
import time

In [ ]:
# ================= 配置参数 =================
CONFIG = {
    'file_path': r'D:\MyNewSynologyDrive\Wuhan University of Technology\柏宗乔师兄\数据\岳化二号2025_01_05.xlsx',
    'seq_len': 30,
    'batch_size': 64,
    'epochs': 300,
    'lr': 0.0001,
    'd_model': 128,
    'nhead': 8,
    'num_layers': 4,
    'dropout': 0.05,
    'sensor_dropout_p': 0.0,
    'patience': 50,
    'alpha': 0.05,  # CP置信水平
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    # CTI权重参数
    'cti_w1': 0.4,  # 精确性权重
    'cti_w2': 0.3,  # 物理一致性权重
    'cti_w3': 0.3   # 不确定性质量权重
}

print(f">>> 运行设备: {CONFIG['device']}")
print(f">>> CP置信水平: {(1-CONFIG['alpha'])*100:.0f}%")
print(f">>> CTI权重设置: w1={CONFIG['cti_w1']}, w2={CONFIG['cti_w2']}, w3={CONFIG['cti_w3']}")

In [ ]:
# ================= 数据加载 =================
def load_and_process_data(file_path):
    print(">>> 正在加载数据并构建物理特征...")
    df = pd.read_excel(file_path)
    df.columns = [c.strip() for c in df.columns]
    
    col_radar = '1#雷达液位(mm)'
    col_pressure = '1#舱压力（Mpa)' 
    col_temp_top = '1#舱顶部温度（℃）'
    col_temp_mid = '1#舱中部温度（℃）'
    col_temp_bot = '1#舱底部温度（℃）'

    df['P_abs'] = df[col_pressure] + 0.1013
    df['T_gas_K'] = df[col_temp_top] + 273.15
    df['Gas_Factor'] = df['T_gas_K'] / (df['P_abs'] + 1e-6)
    df['Delta_T_Vertical'] = df[col_temp_top] - df[col_temp_bot]
    df['Temp_Gradient'] = (df[col_temp_top] - df[col_temp_mid]) / \
                          (df[col_temp_mid] - df[col_temp_bot] + 1e-6)
    
    feature_cols = [
        col_radar, col_pressure, col_temp_top, col_temp_mid, col_temp_bot,
        'Delta_T_Vertical', 'Temp_Gradient', 'Gas_Factor'
    ]
    
    data = df[feature_cols].dropna().values
    return data, feature_cols

In [ ]:
# ================= 数据集构建 =================
def create_dataset(data, seq_len):
    X, X_phys, Y = [], [], []
    for i in range(len(data) - seq_len):
        X.append(data[i : i + seq_len])
        X_phys.append(data[i + seq_len - 1, -1])
        Y.append(data[i + seq_len, 0])
    return np.array(X), np.array(X_phys).reshape(-1, 1), np.array(Y).reshape(-1, 1)

def prepare_data_split(train_data, val_data, test_data, seq_len):
    X_train, X_phys_train, Y_train = create_dataset(train_data, seq_len)
    X_val, X_phys_val, Y_val = create_dataset(val_data, seq_len)
    X_test, X_phys_test, Y_test = create_dataset(test_data, seq_len)
    
    train_loader = DataLoader(
        TensorDataset(torch.Tensor(X_train), torch.Tensor(Y_train), torch.Tensor(X_phys_train)),
        batch_size=CONFIG['batch_size'], shuffle=True
    )
    val_loader = DataLoader(
        TensorDataset(torch.Tensor(X_val), torch.Tensor(Y_val), torch.Tensor(X_phys_val)),
        batch_size=CONFIG['batch_size'], shuffle=False
    )
    test_loader = DataLoader(
        TensorDataset(torch.Tensor(X_test), torch.Tensor(Y_test), torch.Tensor(X_phys_test)),
        batch_size=CONFIG['batch_size'], shuffle=False
    )
    
    print(f"序列构建 -> 训练: {len(X_train)}, 验证: {len(X_val)}, 测试: {len(X_test)}")
    return train_loader, val_loader, test_loader, X_phys_test

In [ ]:
# ================= 模型定义 =================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    
    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class SensorDropout(nn.Module):
    def __init__(self, p=0.3):
        super().__init__()
        self.p = p
    
    def forward(self, x):
        if self.training and self.p > 0:
            mask = torch.rand(x.size(0), x.size(1), 1, device=x.device) > self.p
            feature_mask = torch.ones_like(x)
            feature_mask[:, :, 0] = mask.squeeze(-1)
            return x * feature_mask
        return x

class PhysicsConstraintLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(1, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
    
    def forward(self, gas_factor):
        return self.mlp(gas_factor)

class PhysicsResidualTransformer(nn.Module):
    def __init__(self, input_dim, d_model, nhead, num_layers, dropout, sensor_dropout_p):
        super().__init__()
        self.phys_adapter = PhysicsConstraintLayer()
        self.sensor_dropout = SensorDropout(p=sensor_dropout_p)
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*2,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.dropout_layer = nn.Dropout(dropout)
        self.head_mu = nn.Linear(d_model, 1)
    
    def forward(self, x, x_phys):
        base_level = self.phys_adapter(x_phys)
        x = self.sensor_dropout(x)
        feat = self.input_proj(x)
        feat = self.pos_encoder(feat)
        feat = self.transformer(feat)
        feat_last = self.dropout_layer(feat[:, -1, :])
        mu_residual = self.head_mu(feat_last)
        final_pred = base_level + mu_residual
        return final_pred

In [ ]:
# ================= Loss与工具 =================
class EarlyStopping:
    def __init__(self, patience=20, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float('inf')
        self.early_stop = False
        self.best_state = None
    
    def __call__(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.best_state = copy.deepcopy(model.state_dict())
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
    
    def restore_best_weights(self, model):
        if self.best_state:
            model.load_state_dict(self.best_state)
            print(f"已恢复最佳模型，最佳验证Loss: {self.best_loss:.4f}")

In [ ]:
# ================= Conformal Prediction核心实现 =================

class ConformalPredictor:
    def __init__(self, alpha=0.05):
        self.alpha = alpha
        self.q_hat = None
        self.calibration_scores = None
    
    def calibrate(self, y_true, y_pred):
        scores = np.abs(y_true - y_pred).flatten()
        self.calibration_scores = scores
        n = len(scores)
        q_level = np.ceil((n + 1) * (1 - self.alpha)) / n
        self.q_hat = np.quantile(scores, q_level)
        
        print(f"\n>>> Conformal Prediction校准完成")
        print(f"  校准集大小: {n}")
        print(f"  置信水平: {(1-self.alpha)*100:.0f}%")
        print(f"  分位数阈值: {self.q_hat:.2f} mm")
        print(f"  校准集平均误差: {np.mean(scores):.2f} mm")
    
    def predict(self, y_pred):
        if self.q_hat is None:
            raise ValueError("必须先调用calibrate()进行校准")
        lower = y_pred - self.q_hat
        upper = y_pred + self.q_hat
        return lower, upper
    
    def get_interval_width(self):
        return 2 * self.q_hat if self.q_hat is not None else None

In [ ]:
# ================= 预测与评估 =================

def get_predictions(model, loader, scaler_y):
    """获取模型预测值"""
    model.eval()
    all_preds, all_truths = [], []
    
    with torch.no_grad():
        for batch_x, batch_y, batch_p in loader:
            batch_x = batch_x.to(CONFIG['device'])
            batch_p = batch_p.to(CONFIG['device'])
            batch_y = batch_y.to(CONFIG['device'])
            
            pred = model(batch_x, batch_p)
            all_preds.append(pred.cpu().numpy())
            all_truths.append(batch_y.cpu().numpy())
    
    all_preds = np.concatenate(all_preds, axis=0)
    all_truths = np.concatenate(all_truths, axis=0)
    
    pred_mm = scaler_y.inverse_transform(all_preds)
    truth_mm = scaler_y.inverse_transform(all_truths)
    
    return pred_mm, truth_mm

def calculate_metrics(y_true, y_pred, lower, upper, confidence=0.95):
    """计算评估指标"""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    wmape = np.sum(np.abs(y_true - y_pred)) / (np.sum(np.abs(y_true)) + 1e-8) * 100
    
    hits = (y_true >= lower) & (y_true <= upper)
    picp = np.mean(hits) * 100
    mpiw = np.mean(upper - lower)
    
    eta = 50
    if picp < confidence * 100:
        cwc = mpiw * (1 + np.exp(eta * (confidence - picp/100)))
    else:
        cwc = mpiw
    
    return {
        'RMSE': rmse, 'MAE': mae, 'R2': r2, 'WMAPE': wmape,
        'PICP': picp, 'MPIW': mpiw, 'CWC': cwc
    }

def print_metrics(metrics, method_name="Conformal Prediction"):
    print("\n" + "=" * 50)
    print(f"   LPG储罐虚拟传感器 - {method_name}")
    print("=" * 50)
    print("【A. 预测精度】")
    print(f"  R² Score  : {metrics['R2']:.4f}  (>0.95为优)")
    print(f"  RMSE      : {metrics['RMSE']:.2f} mm")
    print(f"  MAE       : {metrics['MAE']:.2f} mm")
    print(f"  WMAPE     : {metrics['WMAPE']:.2f}%")
    print("-" * 50)
    print("【B. 可信度评估（CP保证）】")
    print(f"  PICP (覆盖率)      : {metrics['PICP']:.2f}% (目标: 95%)")
    print(f"  MPIW (平均区间)    : {metrics['MPIW']:.2f} mm")
    print(f"  CWC (综合指标)     : {metrics['CWC']:.2f}")
    print("-" * 50)
    print("【C. 智能诊断】")
    if 94 <= metrics['PICP'] <= 96:
        print("  ✅ 完美：覆盖率精准，符合理论保证")
    elif metrics['PICP'] < 90:
        print("  ❌ 警告：覆盖率不足")
    elif metrics['PICP'] > 98:
        print("  ⚠️  提示：覆盖率过高，区间可能偏保守")
    print("=" * 50 + "\n")

In [ ]:
# ================= 综合可信度指数 (CTI) 计算 =================

def calculate_cti(y_true, y_pred, lower, upper, phys_param_true, phys_param_pred, 
                  alpha=0.05, w1=0.4, w2=0.3, w3=0.3):
    """
    计算综合可信度指数 (Composite Trustworthiness Index)
    
    Parameters:
    -----------
    y_true : array, 真实值
    y_pred : array, 预测值
    lower : array, 预测区间下界
    upper : array, 预测区间上界
    phys_param_true : array, 真实物理信息参数 (如Gas_Factor)
    phys_param_pred : array, 预测对应的物理信息参数
    alpha : float, 显著性水平 (默认0.05对应95%置信度)
    w1, w2, w3 : float, 三个维度的权重
    
    Returns:
    --------
    dict : 包含CTI及各子指标的字典
    """
    
    # (1) 精确性得分 S_acc
    # NRMSE = RMSE / (y_max - y_min)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    y_range = np.max(y_true) - np.min(y_true)
    nrmse = rmse / (y_range + 1e-8)
    S_acc = 1 / (1 + nrmse)
    
    # (2) 物理一致性得分 S_phys
    # 计算真实值与物理参数的Spearman相关系数
    rho_true, _ = spearmanr(y_true.flatten(), phys_param_true.flatten())
    # 计算预测值与物理参数的Spearman相关系数
    rho_pred, _ = spearmanr(y_pred.flatten(), phys_param_pred.flatten())
    # 物理一致性得分
    S_phys = 1 - np.abs(rho_true - rho_pred)
    
    # (3) 不确定性质量得分 S_uq
    # 覆盖率
    hits = (y_true >= lower) & (y_true <= upper)
    gamma = np.mean(hits)  # 实际覆盖率 (0-1之间)
    gamma_target = 1 - alpha  # 目标覆盖率
    
    # 归一化区间宽度
    mpiw = np.mean(upper - lower)
    R_global = y_range  # 全局极差
    W_norm = mpiw / (R_global + 1e-8)
    
    # 不确定性质量得分
    S_uq = gamma * (1 - W_norm)
    
    # (4) 综合可信度指数 CTI
    CTI = w1 * S_acc + w2 * S_phys + w3 * S_uq
    
    return {
        'CTI': CTI,
        'S_acc': S_acc,
        'S_phys': S_phys,
        'S_uq': S_uq,
        'NRMSE': nrmse,
        'rho_true': rho_true,
        'rho_pred': rho_pred,
        'gamma': gamma,
        'gamma_target': gamma_target,
        'W_norm': W_norm,
        'weights': {'w1': w1, 'w2': w2, 'w3': w3}
    }

def print_cti_report(cti_results):
    """
    打印CTI详细报告
    """
    print("\n" + "=" * 60)
    print("         综合可信度指数 (CTI) 评估报告")
    print("=" * 60)
    
    # 主要指标
    print("\n【总体评分】")
    print(f"  CTI (综合可信度指数): {cti_results['CTI']:.4f}")
    print(f"  评级: {get_cti_rating(cti_results['CTI'])}")
    
    # 三个维度详细得分
    print("\n" + "-" * 60)
    print("【维度一: 精确性 (Accuracy)】")
    print(f"  得分 S_acc: {cti_results['S_acc']:.4f}")
    print(f"  NRMSE: {cti_results['NRMSE']:.4f}")
    print(f"  诊断: {get_accuracy_diagnosis(cti_results['S_acc'])}")
    
    print("\n" + "-" * 60)
    print("【维度二: 物理一致性 (Physics Consistency)】")
    print(f"  得分 S_phys: {cti_results['S_phys']:.4f}")
    print(f"  真实值相关系数 ρ_true: {cti_results['rho_true']:.4f}")
    print(f"  预测值相关系数 ρ_pred: {cti_results['rho_pred']:.4f}")
    print(f"  相关系数差异: {np.abs(cti_results['rho_true'] - cti_results['rho_pred']):.4f}")
    print(f"  诊断: {get_physics_diagnosis(cti_results['S_phys'])}")
    
    print("\n" + "-" * 60)
    print("【维度三: 不确定性质量 (UQ Reliability)】")
    print(f"  得分 S_uq: {cti_results['S_uq']:.4f}")
    print(f"  实际覆盖率 γ: {cti_results['gamma']*100:.2f}%")
    print(f"  目标覆盖率: {cti_results['gamma_target']*100:.2f}%")
    print(f"  归一化区间宽度 W_norm: {cti_results['W_norm']:.4f}")
    print(f"  诊断: {get_uq_diagnosis(cti_results['S_uq'], cti_results['gamma'], cti_results['gamma_target'])}")
    
    # 权重信息
    print("\n" + "-" * 60)
    print("【权重设置】")
    w = cti_results['weights']
    print(f"  精确性权重 w1: {w['w1']}")
    print(f"  物理一致性权重 w2: {w['w2']}")
    print(f"  不确定性质量权重 w3: {w['w3']}")
    
    print("\n" + "=" * 60)
    print("【综合建议】")
    print(get_overall_suggestions(cti_results))
    print("=" * 60 + "\n")

def get_cti_rating(cti):
    """根据CTI值返回评级"""
    if cti >= 0.90:
        return "⭐⭐⭐⭐⭐ 优秀 (Excellent)"
    elif cti >= 0.80:
        return "⭐⭐⭐⭐ 良好 (Good)"
    elif cti >= 0.70:
        return "⭐⭐⭐ 中等 (Fair)"
    elif cti >= 0.60:
        return "⭐⭐ 及格 (Pass)"
    else:
        return "⭐ 需改进 (Needs Improvement)"

def get_accuracy_diagnosis(s_acc):
    """精确性诊断"""
    if s_acc >= 0.95:
        return "✅ 优秀: 预测精度极高"
    elif s_acc >= 0.90:
        return "✅ 良好: 预测精度较高"
    elif s_acc >= 0.80:
        return "⚠️  中等: 预测精度一般，建议优化模型"
    else:
        return "❌ 较差: 预测精度低，需要重新训练"

def get_physics_diagnosis(s_phys):
    """物理一致性诊断"""
    if s_phys >= 0.95:
        return "✅ 优秀: 预测完全符合物理规律"
    elif s_phys >= 0.90:
        return "✅ 良好: 预测基本符合物理规律"
    elif s_phys >= 0.80:
        return "⚠️  中等: 存在一定物理不一致性"
    else:
        return "❌ 较差: 预测违背物理规律，需增强物理约束"

def get_uq_diagnosis(s_uq, gamma, gamma_target):
    """不确定性质量诊断"""
    coverage_ok = abs(gamma - gamma_target) < 0.02  # 覆盖率在目标±2%内
    
    if s_uq >= 0.90 and coverage_ok:
        return "✅ 优秀: 区间精确且窄，可信度高"
    elif s_uq >= 0.85:
        return "✅ 良好: 不确定性估计可靠"
    elif s_uq >= 0.75:
        if gamma < gamma_target:
            return "⚠️  提示: 覆盖率偏低，区间可能过窄"
        else:
            return "⚠️  提示: 区间较宽，可尝试提高模型确定性"
    else:
        return "❌ 较差: 不确定性估计不可靠"

def get_overall_suggestions(cti_results):
    """综合建议"""
    suggestions = []
    
    # 精确性建议
    if cti_results['S_acc'] < 0.85:
        suggestions.append("• 提高精确性: 考虑增加训练数据、优化超参数或使用更复杂的模型")
    
    # 物理一致性建议
    if cti_results['S_phys'] < 0.85:
        suggestions.append("• 增强物理一致性: 加强物理约束层、引入更多物理知识或增加物理损失项")
    
    # 不确定性质量建议
    if cti_results['S_uq'] < 0.85:
        if cti_results['gamma'] < cti_results['gamma_target']:
            suggestions.append("• 改进不确定性估计: 覆盖率不足，考虑使用更保守的校准方法")
        else:
            suggestions.append("• 优化区间宽度: 当前区间过宽，可尝试使用自适应CP或其他UQ方法")
    
    if not suggestions:
        return "✅ 模型表现优异，各项指标均达标！"
    else:
        return "\n".join(suggestions)

In [ ]:
# ================= 可视化 =================
def plot_results(history, pred, truth, lower, upper):
    fig = plt.figure(figsize=(15, 10))
    
    # 1. Loss曲线
    ax1 = plt.subplot(3, 2, 1)
    ax1.plot(history['train_loss'], label='Train Loss', linewidth=2)
    ax1.plot(history['val_loss'], label='Val Loss', linewidth=2)
    ax1.set_title('Training Curve (MSE Loss)')
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(alpha=0.3)
    
    # 2. 预测区间宽度分布
    ax2 = plt.subplot(3, 2, 2)
    widths = (upper - lower).flatten()
    ax2.hist(widths, bins=50, alpha=0.7, edgecolor='black')
    ax2.axvline(np.mean(widths), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(widths):.1f}mm')
    ax2.set_title('CP Interval Width Distribution')
    ax2.set_xlabel('Width (mm)')
    ax2.set_ylabel('Frequency')
    ax2.legend()
    ax2.grid(alpha=0.3)
    
    # 3. 预测对比
    ax3 = plt.subplot(3, 2, 3)
    n_show = min(500, len(pred))
    ax3.plot(truth[:n_show], label='True', alpha=0.7, linewidth=1.5)
    ax3.plot(pred[:n_show], label='Predicted', alpha=0.7, linewidth=1.5)
    ax3.fill_between(range(n_show),
                     lower[:n_show].flatten(),
                     upper[:n_show].flatten(),
                     alpha=0.2, label='95% CP Interval')
    ax3.set_title('Prediction vs Truth with CP Intervals')
    ax3.set_xlabel('Time Steps')
    ax3.set_ylabel('Liquid Level (mm)')
    ax3.legend()
    ax3.grid(alpha=0.3)
    
    # 4. 散点图
    ax4 = plt.subplot(3, 2, 4)
    ax4.scatter(truth, pred, alpha=0.3, s=5)
    ax4.plot([truth.min(), truth.max()], [truth.min(), truth.max()],
             'r--', linewidth=2, label='Perfect')
    ax4.set_title('Predicted vs True')
    ax4.set_xlabel('True (mm)')
    ax4.set_ylabel('Predicted (mm)')
    ax4.legend()
    ax4.grid(alpha=0.3)
    
    # 5. 覆盖率随时间变化
    ax5 = plt.subplot(3, 2, 5)
    window = 100
    coverage = []
    for i in range(0, len(pred) - window, 10):
        hits = ((truth[i:i+window] >= lower[i:i+window]) & 
                (truth[i:i+window] <= upper[i:i+window]))
        coverage.append(np.mean(hits) * 100)
    ax5.plot(coverage, alpha=0.7, linewidth=2)
    ax5.axhline(95, color='red', linestyle='--', linewidth=2, label='Target 95%')
    ax5.set_title('Coverage Rate Over Time')
    ax5.set_xlabel('Window Index')
    ax5.set_ylabel('Coverage (%)')
    ax5.legend()
    ax5.grid(alpha=0.3)
    
    # 6. 误差分布
    ax6 = plt.subplot(3, 2, 6)
    errors = (pred - truth).flatten()
    ax6.hist(errors, bins=50, alpha=0.7, edgecolor='black')
    ax6.axvline(0, color='red', linestyle='--', linewidth=2)
    ax6.set_title('Prediction Error Distribution')
    ax6.set_xlabel('Error (mm)')
    ax6.set_ylabel('Frequency')
    ax6.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def plot_cti_breakdown(cti_results):
    """可视化CTI各维度得分"""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 子图1: 雷达图
    ax1 = axes[0]
    categories = ['精确性\n(Accuracy)', '物理一致性\n(Physics)', '不确定性\n(UQ)']
    values = [cti_results['S_acc'], cti_results['S_phys'], cti_results['S_uq']]
    
    angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
    values += values[:1]
    angles += angles[:1]
    
    ax1 = plt.subplot(121, projection='polar')
    ax1.plot(angles, values, 'o-', linewidth=2, label=f'CTI = {cti_results["CTI"]:.3f}')
    ax1.fill(angles, values, alpha=0.25)
    ax1.set_xticks(angles[:-1])
    ax1.set_xticklabels(categories)
    ax1.set_ylim(0, 1)
    ax1.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax1.legend(loc='upper right')
    ax1.set_title('CTI 三维度雷达图', pad=20)
    ax1.grid(True)
    
    # 子图2: 柱状图
    ax2 = axes[1]
    x = np.arange(len(categories))
    bars = ax2.bar(x, [cti_results['S_acc'], cti_results['S_phys'], cti_results['S_uq']], 
                   color=['#2E86AB', '#A23B72', '#F18F01'], alpha=0.7, edgecolor='black')
    ax2.axhline(y=0.85, color='green', linestyle='--', linewidth=2, label='优秀线 (0.85)')
    ax2.set_xticks(x)
    ax2.set_xticklabels(categories)
    ax2.set_ylabel('得分')
    ax2.set_ylim(0, 1.0)
    ax2.set_title(f'CTI维度得分对比 (总分: {cti_results["CTI"]:.3f})')
    ax2.legend()
    ax2.grid(axis='y', alpha=0.3)
    
    # 在柱子上显示数值
    for bar in bars:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# ================= 主程序 =================
def main():
    # 1. 加载数据
    raw_data, feature_cols = load_and_process_data(CONFIG['file_path'])
    print(f"特征列: {feature_cols}")
    
    # 2. 划分数据集
    n = len(raw_data)
    train_end = int(n * 0.7)
    val_end = int(n * 0.85)
    
    train_data_raw = raw_data[:train_end]
    val_data_raw = raw_data[train_end:val_end]
    test_data_raw = raw_data[val_end:]
    
    print(f"原始数据划分 -> 训练: {len(train_data_raw)}, 验证: {len(val_data_raw)}, 测试: {len(test_data_raw)}")
    
    # 3. 归一化
    scaler = MinMaxScaler()
    scaler.fit(train_data_raw)
    
    scaler_y = MinMaxScaler()
    scaler_y.fit(train_data_raw[:, 0:1])
    
    train_norm = scaler.transform(train_data_raw)
    val_norm = scaler.transform(val_data_raw)
    test_norm = scaler.transform(test_data_raw)
    
    # 4. 准备数据集 (保存物理参数用于CTI计算)
    train_loader, val_loader, test_loader, test_phys_param = prepare_data_split(
        train_norm, val_norm, test_norm, CONFIG['seq_len']
    )
    
    # 5. 初始化模型
    model = PhysicsResidualTransformer(
        input_dim=len(feature_cols),
        d_model=CONFIG['d_model'],
        nhead=CONFIG['nhead'],
        num_layers=CONFIG['num_layers'],
        dropout=CONFIG['dropout'],
        sensor_dropout_p=CONFIG['sensor_dropout_p']
    ).to(CONFIG['device'])
    
    print(f"模型参数量: {sum(p.numel() for p in model.parameters()):,}")
    
    # 6. 训练
    optimizer = optim.Adam(model.parameters(), lr=CONFIG['lr'], weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=10, factor=0.5, verbose=True
    )
    early_stopping = EarlyStopping(patience=CONFIG['patience'])
    
    print("\n>>> 开始训练...\n")
    start_time = time.time()
    history = {'train_loss': [], 'val_loss': []}
    
    for epoch in range(CONFIG['epochs']):
        # 训练
        model.train()
        train_loss_sum = 0
        for bx, by, bp in train_loader:
            bx, by, bp = bx.to(CONFIG['device']), by.to(CONFIG['device']), bp.to(CONFIG['device'])
            optimizer.zero_grad()
            pred = model(bx, bp)
            loss = nn.MSELoss()(pred, by)
            loss.backward()
            optimizer.step()
            train_loss_sum += loss.item()
        
        avg_train_loss = train_loss_sum / len(train_loader)
        
        # 验证
        model.eval()
        val_loss_sum = 0
        with torch.no_grad():
            for bx, by, bp in val_loader:
                bx, by, bp = bx.to(CONFIG['device']), by.to(CONFIG['device']), bp.to(CONFIG['device'])
                pred = model(bx, bp)
                loss = nn.MSELoss()(pred, by)
                val_loss_sum += loss.item()
        
        avg_val_loss = val_loss_sum / len(val_loader)
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        
        if (epoch + 1) % 5 == 0 or epoch < 10:
            print(f"Epoch {epoch+1:03d} | Train: {avg_train_loss:.5f} | Val: {avg_val_loss:.5f}")
        
        scheduler.step(avg_val_loss)
        early_stopping(avg_val_loss, model)
        
        if early_stopping.early_stop:
            print(f"\n🚀 触发早停！在第{epoch+1}轮停止训练")
            break
    
    early_stopping.restore_best_weights(model)
    train_time = time.time() - start_time
    print(f"训练耗时: {train_time:.2f} 秒")
    
    # 7. Conformal Prediction校准
    print("\n>>> 在验证集上校准Conformal Prediction...")
    val_pred, val_truth = get_predictions(model, val_loader, scaler_y)
    
    cp = ConformalPredictor(alpha=CONFIG['alpha'])
    cp.calibrate(val_truth, val_pred)
    
    # 8. 测试集预测
    print("\n>>> 在测试集上进行预测...")
    test_pred, test_truth = get_predictions(model, test_loader, scaler_y)
    test_lower, test_upper = cp.predict(test_pred)
    
    # 9. 基础评估指标
    metrics = calculate_metrics(
        test_truth.flatten(), test_pred.flatten(),
        test_lower.flatten(), test_upper.flatten()
    )
    print_metrics(metrics, "Conformal Prediction")
    
    # 10. 反归一化物理参数用于CTI计算
    # 注意: Gas_Factor是第8列(索引7),需要从scaler中提取对应的缩放参数
    phys_param_scaler = MinMaxScaler()
    phys_param_scaler.fit(test_data_raw[:, -1].reshape(-1, 1))
    test_phys_param_denorm = phys_param_scaler.inverse_transform(test_phys_param)
    
    # 对于预测值,我们使用相同的物理参数(因为是在相同时间点)
    pred_phys_param = test_phys_param_denorm.copy()
    
    # 11. 计算CTI
    print("\n>>> 计算综合可信度指数 (CTI)...")
    cti_results = calculate_cti(
        y_true=test_truth,
        y_pred=test_pred,
        lower=test_lower,
        upper=test_upper,
        phys_param_true=test_phys_param_denorm,
        phys_param_pred=pred_phys_param,
        alpha=CONFIG['alpha'],
        w1=CONFIG['cti_w1'],
        w2=CONFIG['cti_w2'],
        w3=CONFIG['cti_w3']
    )
    
    # 12. 打印CTI报告
    print_cti_report(cti_results)
    
    # 13. 可视化
    print(">>> 生成可视化结果...")
    plot_results(history, test_pred, test_truth, test_lower, test_upper)
    plot_cti_breakdown(cti_results)
    
    print("✅ 评估完成！")
    
    # 14. 返回结果
    results = {
        **metrics,
        'history': history,
        'train_time': train_time,
        'y_true': test_truth,
        'y_pred': test_pred,
        'lower': test_lower,
        'upper': test_upper,
        'phys_param': test_phys_param_denorm,
        'cti_results': cti_results  # 新增CTI结果
    }
    
    return results

In [ ]:
# ========== 运行实验 ==========
results = main()

# 检查结果
print("\n>>> Results包含以下内容:")
for key in results.keys():
    if key == 'cti_results':
        print(f"  - {key}: {list(results[key].keys())}")
    else:
        print(f"  - {key}")

In [ ]:
# ========== 保存结果(包含CTI) ==========
import pickle
from pathlib import Path
from datetime import datetime

SAVE_DIR = Path(r"D:\MyNewSynologyDrive\jupyter\Virtual Physical Fusion\Phys-Trans-CP-LPG\results")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
experiment_name = 'with_CTI'

print(f"Saving {experiment_name}...")

save_data = {
    'experiment_name': experiment_name,
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'model_type': 'Physics-Informed Transformer with CTI',
    'metrics': {
        'R2': float(results['R2']),
        'RMSE': float(results['RMSE']),
        'MAE': float(results['MAE']),
        'PICP': float(results['PICP']),
        'MPIW': float(results['MPIW']),
        'train_time': float(results['train_time']),
    },
    'cti_metrics': results['cti_results'],  # 新增CTI指标
    'history': results['history'],
    'predictions': {
        'y_true': results['y_true'],
        'y_pred': results['y_pred'],
        'lower': results['lower'],
        'upper': results['upper'],
        'phys_param': results['phys_param'],
        'n_samples': len(results['y_true'])
    },
    'config': CONFIG
}

# 保存PKL
with open(SAVE_DIR / f'{experiment_name}_results.pkl', 'wb') as f:
    pickle.dump(save_data, f)
print(f"✓ Saved: {experiment_name}_results.pkl")

# 保存CSV
df = pd.DataFrame({
    'y_true': results['y_true'].flatten(),
    'y_pred': results['y_pred'].flatten(),
    'lower': results['lower'].flatten(),
    'upper': results['upper'].flatten(),
    'phys_param': results['phys_param'].flatten()
})
df.to_csv(SAVE_DIR / f'{experiment_name}_predictions.csv', index=False)
print(f"✓ Saved: {experiment_name}_predictions.csv")

# 保存CTI报告为文本
with open(SAVE_DIR / f'{experiment_name}_cti_report.txt', 'w', encoding='utf-8') as f:
    f.write("综合可信度指数 (CTI) 评估报告\n")
    f.write("=" * 60 + "\n\n")
    f.write(f"CTI总分: {results['cti_results']['CTI']:.4f}\n")
    f.write(f"  - 精确性得分 S_acc: {results['cti_results']['S_acc']:.4f}\n")
    f.write(f"  - 物理一致性得分 S_phys: {results['cti_results']['S_phys']:.4f}\n")
    f.write(f"  - 不确定性质量得分 S_uq: {results['cti_results']['S_uq']:.4f}\n")
    f.write(f"\nNRMSE: {results['cti_results']['NRMSE']:.4f}\n")
    f.write(f"Spearman ρ (真实): {results['cti_results']['rho_true']:.4f}\n")
    f.write(f"Spearman ρ (预测): {results['cti_results']['rho_pred']:.4f}\n")
    f.write(f"覆盖率 γ: {results['cti_results']['gamma']*100:.2f}%\n")
    f.write(f"归一化区间宽度: {results['cti_results']['W_norm']:.4f}\n")
print(f"✓ Saved: {experiment_name}_cti_report.txt")

print(f"\n✅ Done! Saved {len(df)} predictions with CTI metrics")

## CTI指标说明

综合可信度指数(CTI)由三个维度组成:

1. **精确性 (Accuracy)**: 衡量预测值与真实值的接近程度
   - 基于NRMSE计算: S_acc = 1/(1+NRMSE)

2. **物理一致性 (Physics Consistency)**: 衡量预测是否符合物理规律
   - 基于Spearman相关系数: S_phys = 1 - |ρ_true - ρ_pred|

3. **不确定性质量 (UQ Reliability)**: 衡量预测区间的可靠性和效率
   - S_uq = γ × (1 - W_norm)
   - γ: 实际覆盖率
   - W_norm: 归一化区间宽度

**CTI总分**: CTI = w1×S_acc + w2×S_phys + w3×S_uq

权重可根据应用场景调整。默认设置: w1=0.4, w2=0.3, w3=0.3